
# Agentic Multimodal Demo (Notebook)
Lightweight, agentic pipeline you can run step-by-step:
- **SeriesBuilder**: ordered people/things with dates (kings, CEOs, etc.) → SDXL portraits → poster
- **MapBuilder**: regions/points (Europe, US states, Canadian provinces, South America) → flags/icons → map

Repo layout assumption:
```
/notebooks/agentic_multimodal/agentic-multimodal.ipynb      # this notebook (root)
/src/agentic_multimodal #supporting modules
/results/agentic_multimodal    # output data
```


In [1]:

# --- Optional installs (uncomment as needed) ---
# %pip install langgraph langchain pydantic diffusers transformers accelerate
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install pillow geopandas shapely requests matplotlib bs4 pyproj
# %pip install ipywidgets
# Note: On Macs, torch install line will differ (MPS). See PyTorch docs.


#### Load all the relevant code into a registry

In [2]:
import pathlib as p
from agentic_multimodal.core.config import CACHE, RESULTS, SRC
from agentic_multimodal.schemas.artifacts import PosterSpec, PosterItem, ImageAsset
from agentic_multimodal.services.registry import make_registry
from agentic_multimodal.skills.adapters import name_year_pairs, countries_to_mapspec
from agentic_multimodal.skills.image_gen import generate_person_images
from agentic_multimodal.skills.gen_poster_renderer import compose_poster_spec
import sys
from support import tree_markdown
import os, re

ROOT = p.Path("..").resolve().parent
MANIFEST = ROOT / "results" / "agentic_multimodal" / "manifest.jsonl"
reg = make_registry(ROOT)

BUILD_TEASER = False
BUILD_POTUS = False
BUILD_EUROPE = False
BUILD_NOBEL_PHYSICS = False
BUILD_MONARCHS_ENG_GB_UK = False


In [3]:

for dir in [CACHE, RESULTS]:
    dir.mkdir(parents=True, exist_ok=True)  # ensure directory exists
    if str(dir) not in sys.path:
        sys.path.insert(0, str(dir))

# Show code tree from source:
display(tree_markdown(SRC))

```agentic_multimodal/
├── assets
│   ├── ne
│   │   ├── ne_110m_admin_0_countries.geojson
│   │   └── ne_50m_admin_0_countries.geojson
│   └── prompts
│       └── overrides.yaml
├── cache
├── core
│   ├── __init__.py
│   └── config.py
├── graphs
│   ├── __init__.py
│   ├── factory.py
│   ├── geo_flow.py
│   └── person_flow.py
├── schemas
│   ├── __init__.py
│   ├── artifacts.py
│   ├── entities.py
│   ├── messages.py
│   └── requests.py
├── services
│   ├── cache.py
│   ├── io_web_fetcher.py
│   ├── llm_factory.py
│   ├── registry.py
│   └── settings.py
├── skills
│   ├── adapters
│   │   ├── __init__.py
│   │   ├── countries_to_map.py
│   │   ├── people_to_poster.py
│   │   └── picks_to_portraits.py
│   ├── data
│   │   ├── __init__.py
│   │   ├── natural_earth.py
│   │   ├── wd_utils.py
│   │   ├── wiki_pageviews.py
│   │   ├── wikidata_geo.py
│   │   ├── wikidata_search_label.py
│   │   ├── wikidata_series.py
│   │   └── wikidata_sparql.py
│   ├── geo
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── europe_flags.py
│   │   ├── region_flags.py
│   │   └── subdivisions.py
│   ├── selectors
│   │   ├── __init__.py
│   │   └── famous_by_country.py
│   ├── series
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── award.py
│   │   ├── candidates.py
│   │   ├── per_country.py
│   │   ├── positions.py
│   │   └── rankers.py
│   ├── __init__.py
│   ├── gen_map_renderer.py
│   ├── gen_poster_renderer.py
│   ├── image_gen.py
│   ├── image_prompts.py
│   └── map_regions.py
├── __init__.py
└── notebook_utils.py
```

## Poster compositor (grid + captions)
### Build teaser image of a handful of presidents

In [4]:
if BUILD_POTUS:
    reg = make_registry(ROOT)
    people = reg.series.run("potus")
    pairs = name_year_pairs(people, mode="per_term")
    paths = generate_person_images(pairs, outdir="artifacts/potus_terms_portraits")
    spec  = reg.adapters.people_per_term(people, title="U.S. Presidents — Terms", image_paths=paths, cols=6)
    reg.render.poster(spec, outpath="artifacts/poster_presidents_terms.webp")

In [5]:
if BUILD_NOBEL_PHYSICS:
    laureates = reg.series.run("nobel_physics")
    pairs = name_year_pairs(laureates, mode="per_person_auto")
    paths = generate_person_images(pairs, outdir="artifacts/nobel_physics_portraits")
    spec  = reg.adapters.people_per_person(laureates, title="Nobel Prize in Physics — Laureates", image_paths=paths, cols=12)
    reg.render.poster(spec, outpath="artifacts/poster_nobel_physics.webp")


In [6]:
if BUILD_MONARCHS_ENG_GB_UK:
    reg = make_registry(ROOT)
    people = reg.series.run("monarchs_eng_gb_uk")
    pairs = name_year_pairs(people, mode="per_term")
    paths = generate_person_images(pairs, outdir="artifacts/monarchs_terms_portraits")
    spec  = reg.adapters.people_per_term(people, title="UK Monarchs — Terms", image_paths=paths, cols=8)
    reg.render.poster(spec, outpath="artifacts/poster_monarchs_terms.webp")

In [7]:
# 3×2 teaser (names + all term ranges)

if BUILD_TEASER:
    reg = make_registry(ROOT)

    teaser_names = [
        "George Washington",
        "Abraham Lincoln",
        "John F. Kennedy",
        "Ronald Reagan",
        "Barack Obama",
        "Donald Trump",
    ]

    # 1) Look up term years from your series data
    people = reg.series.run("potus")

    def _y(s):
        if not s: return None
        s = s.lstrip("+")
        return s[:4] if len(s) >= 4 and s[:4].isdigit() else None

    def term_spans(person):
        # Build "YYYY–YYYY" or "YYYY–PT"; sort by start year
        spans = []
        for t in getattr(person, "terms", []):
            ys = _y(getattr(t, "start", None))
            ye = _y(getattr(t, "end", None))
            if ys or ye:
                spans.append((ys, ye))
        # sort by numeric start, then end
        spans.sort(key=lambda ab: (int(ab[0]) if (ab[0] and ab[0].isdigit()) else 9999,
                                int(ab[1]) if (ab[1] and ab[1].isdigit()) else 9999))
        # format with en dash
        out = []
        for ys, ye in spans:
            if ys and ye:
                out.append(f"{ys}–{ye}")
            elif ys and not ye:
                out.append(f"{ys}–PT")
            elif not ys and ye:
                out.append(f"–{ye}")
        return ", ".join(out)

    by_name = {p.name: p for p in people}

    # 2) Map names → existing portrait paths (first match per name)
    IMG_DIR = "artifacts/potus_terms_portraits"   # adjust if needed

    def _canon(s): return re.sub(r"\W+", "", s).lower()
    paths_by_name = {}
    for fn in sorted(os.listdir(IMG_DIR)):
        if not fn.lower().endswith((".png",".jpg",".jpeg",".webp")):
            continue
        name_part = re.sub(r"^\d+_", "", os.path.splitext(fn)[0])
        paths_by_name.setdefault(_canon(name_part), os.path.join(IMG_DIR, fn))

    def _portrait_for(name):
        p = paths_by_name.get(_canon(name))
        if not p:
            raise FileNotFoundError(f"No portrait found for '{name}' in {IMG_DIR}")
        return p

    # 3) Build items with NAME + merged term spans
    items = []
    for nm in teaser_names:
        label_lines = [nm]
        if nm in by_name:
            spans = term_spans(by_name[nm])
            if spans:
                label_lines.append(spans)
        label = "\n".join(label_lines)

        items.append(
            PosterItem(
                image=ImageAsset(id=_canon(nm), path=_portrait_for(nm), width=512, height=768),
                label=label
            )
        )

    # 4) 3 columns × 2 rows; export sized for LinkedIn (landscape-ish)
    spec_teaser = PosterSpec(
        title="U.S. Presidents — Agentic Poster (Teaser)",
        grid_cols=3,
        items=items
    )

    out_teaser = "artifacts/linkedin/potus_teaser_3x2_terms_1620x1080.jpg"
    os.makedirs(os.path.dirname(out_teaser), exist_ok=True)

    # Slightly larger captions help on mobile; tune if your renderer exposes these knobs
    compose_poster_spec(
        spec_teaser,
        outpath=out_teaser,
        out_format="JPEG",
        out_quality=88,
        max_long_side=1620,          # ~1620×1080 export
        # If your renderer supports it, these help readability:
        # caption_scale=2.2,
        # line_gap_px=8,
        # stroke_width_px=3,
    )

    print("Teaser saved:", out_teaser)



## MapBuilder: regions + capitals + flags

In [8]:
if BUILD_EUROPE:
    reg = make_registry(ROOT)
    countries = reg.geo.run("europe_countries_flags")

    spec = countries_to_mapspec(
        reg.geo.run("europe_countries_flags"),
        title="Europe — Flags at Capitals",
        region_key="europe_flags",
    )

    reg.render.map(
        spec,
        outdir="artifacts/maps",
        size=(2200, 1320),
        marker_px=42,              
        show_labels=True,        
        show_country_names=True,
        show_capital_names=False,
        min_flag_separation_px=48,
        min_label_separation_px=96,
        max_labels=None,
    )
    print("Saved:", spec.path)


In [ ]:
def image_from_pick_path(country, pick):
    path = portrait_paths.get(country.qid)
    return {"path": path} if path else None


# import the new API
from agentic_multimodal.skills.adapters import (
    countries_to_mapspec,
    image_from_flag_url,
    image_from_pick_url,
    label_country,
    label_country_plus_pick,
    render_portraits_for_picks,
)

reg.adapters.countries_to_mapspec = countries_to_mapspec
reg.adapters.image_from_pick = image_from_pick_url
reg.adapters.image_from_flag = image_from_flag_url
reg.adapters.label_country = label_country
reg.adapters.label_country_plus_pick = label_country_plus_pick
reg.adapters.render_portraits_for_picks = render_portraits_for_picks

countries = reg.geo.run("europe_countries_flags", min_pop=2_000_000)
top10 = sorted(countries, key=lambda c: c.population or 0, reverse=True)[:10]

picks = {}
for c in top10:
    ppl = reg.geo.run(
        "famous_by_country",
        country_qid=c.qid,
        category="sports",
        limit_candidates=60,
        days=90,
        score="blended",   # ← new
    )
    best = ppl[0] if ppl else None
    if best:
        picks[c.qid] = {"qid": best.qid, "label": best.name, "image_url": best.image_url}
# After you compute portrait_paths:
portrait_paths = render_portraits_for_picks(picks, outdir="artifacts/europe_top10_portraits")

# attach local paths back onto the picks you already have
for qid, p in picks.items():
    lp = portrait_paths.get(qid)
    if lp:
        if isinstance(p, dict):
            p["local_image_path"] = lp
        else:
            setattr(p, "local_image_path", lp)  # if it’s a model you control

spec = countries_to_mapspec(
    top10,
    title="Europe — most popular sportsperson (top-10 countries)",
    picks=picks,
    label_fn=label_country_plus_pick,
    image_fn = lambda c, p: {"path": p.get("local_image_path")} \
    if isinstance(p, dict) and p.get("local_image_path") else None
)


img = reg.render.map(
    spec,
    show_country_names=True,
    show_labels=True,
    show_pick_images=True,          # portrait bubbles on the capitals
    pick_image_size_px=125,          # tweak 44–56 if you like
    show_flag_markers=False,        # portraits > flags here
)
display(img)



[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q113575'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Magomed Ozdoyev'}, 'image': {'type': 'uri', 'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/Magomed%20Ozdoyev%20%28May%202024%29.png'}, 'affinity': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '1'}}]
[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q65548'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Kerstin Stegemann'}, 'image': {'type': 'uri', 'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/Kerstin%20Stegemann%203.jpg'}, 'affinity': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '1'}}]
[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q68132'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Gernot Rohr'}, 'image': {'type': 'uri', 'value': 'http://commons.wikimedia.org/wiki/Special:FilePat

Keyword arguments {'add_watermark': False} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

'artifacts/maps/Europe _ most popular sportsperson _top-10 countries_.png'

In [19]:
def image_from_pick_path(country, pick):
    path = portrait_paths.get(country.qid)
    return {"path": path} if path else None


# import the new API
from agentic_multimodal.skills.adapters import (
    countries_to_mapspec,
    image_from_flag_url,
    image_from_pick_url,
    label_country,
    label_country_plus_pick,
    render_portraits_for_picks,
)

reg.adapters.countries_to_mapspec = countries_to_mapspec
reg.adapters.image_from_pick = image_from_pick_url
reg.adapters.image_from_flag = image_from_flag_url
reg.adapters.label_country = label_country
reg.adapters.label_country_plus_pick = label_country_plus_pick
reg.adapters.render_portraits_for_picks = render_portraits_for_picks

countries = reg.geo.run("europe_countries_flags", min_pop=2_000_000)
top15 = sorted(countries, key=lambda c: c.population or 0, reverse=True)[:15]

picks = {}
for c in top15:
    ppl = reg.geo.run(
        "famous_by_country",
        country_qid=c.qid,
        category="musicians",
        limit_candidates=60,
        days=90,
        score="blended",   # ← new
    )
    best = ppl[0] if ppl else None
    if best:
        picks[c.qid] = {"qid": best.qid, "label": best.name, "image_url": best.image_url}
# After you compute portrait_paths:
portrait_paths = render_portraits_for_picks(picks, outdir="artifacts/europe_top15_portraits")

# attach local paths back onto the picks you already have
for qid, p in picks.items():
    lp = portrait_paths.get(qid)
    if lp:
        if isinstance(p, dict):
            p["local_image_path"] = lp
        else:
            setattr(p, "local_image_path", lp)  # if it’s a model you control

spec = countries_to_mapspec(
    top15,
    title="Europe — most popular musician (top-15 countries)",
    picks=picks,
    label_fn=label_country_plus_pick,
    image_fn = lambda c, p: {"path": p.get("local_image_path")} \
    if isinstance(p, dict) and p.get("local_image_path") else None
)


img = reg.render.map(
    spec,
    show_country_names=True,
    show_labels=True,
    show_pick_images=True,          # portrait bubbles on the capitals
    pick_image_size_px=125,          # tweak 44–56 if you like
    show_flag_markers=False,        # portraits > flags here
)
display(img)


[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q21088'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Zedd'}, 'image': {'type': 'uri', 'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/Zedd%20at%20Veld%202017%20%28cropped%29.jpg'}, 'affinity': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '1'}}]
[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q66245'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Oskar Fischinger'}, 'affinity': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer', 'type': 'literal', 'value': '1'}}]
[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q70326'}, 'name': {'xml:lang': 'en', 'type': 'literal', 'value': 'Pierre Beaumarchais'}, 'image': {'type': 'uri', 'value': 'http://commons.wikimedia.org/wiki/Special:FilePath/D%27apr%C3%A8s%20Jean-Marc%20Nattier%2C%20Portrait%20de%20Pierre-Augustin%20Caron%20de%20Beaumarchais%20%28Biblioth%C

Keyword arguments {'add_watermark': False} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

'artifacts/maps/Europe _ most popular musician _top-15 countries_.png'

In [14]:
# 1) Get U.S. states with flags + capital coords (you already registered this alias)
states = reg.geo.run("us_states_flags")  # SubdivisionsByCountryProvider under the hood

# 2) Keep the West. Adjust the list if you want to be stricter/looser.
WEST_ABBR = {
    "WA","OR","CA","NV","ID","MT","WY","UT","CO","AZ","NM","AK","HI"
}
west_states = [s for s in states if getattr(s, "abbrev", None) in WEST_ABBR]

# 3) Build the map spec: labels optional; flags as the marker image
from agentic_multimodal.skills.adapters import countries_to_mapspec, image_from_flag_url, label_country

spec = countries_to_mapspec(
    west_states,
    title="Western U.S. — State Flags",
    picks=None,                              # no per-state “pick” objects
    label_fn=lambda s, _: s.name,            # or 'label_country' if you prefer
    image_fn=lambda s, _: {"url": s.flag_svg_url},  # <- use the flag as marker image
    region="Western U.S."
)

# 4) Render. Portraits off, flags on. Keep dots tiny as fallback.
img_path = reg.render.map(
    spec,
    show_country_names=False,        # hide text if you want a clean flag-only look
    show_labels=False,               # turn on if you want state names under flags
    show_pick_images=False,          # we’re not using portraits here
    show_flag_markers=True,          # <- show the flags
    flag_marker_size_px=42,          # tweak 36–56 based on your canvas
    marker_px=2,                     # tiny dot fallback
)
display(img_path)


'artifacts/maps/Western U.S. _ State Flags.png'

In [16]:
states = reg.geo.run("us_states_flags")
print("count:", len(states))
print("sample:", [(s.name, getattr(s, "abbrev", None), s.capital_coords) for s in states[:5]])


count: 50
sample: [('Alabama', None, (-86.3, 32.3675)), ('Alaska', None, (-134.41972, 58.30194)), ('Arizona', None, (-112.073888888, 33.448333333)), ('Arkansas', None, (-92.288055555, 34.744444444)), ('California', None, (-121.486111111, 38.575277777))]


In [18]:
portrait_paths = reg.adapters.render_portraits_for_picks(picks, outdir="artifacts/europe_top15_musicians")
for qid, p in picks.items():
    path = portrait_paths.get(qid)
    if path:
        p["local_image_path"] = path

spec = reg.adapters.countries_to_mapspec(
    top15,
    title="Europe — most popular musician (top-15 countries)",
    picks=picks,
    label_fn=reg.adapters.label_country_plus_pick,
    image_fn=lambda c,p: ({"path": p.get("local_image_path")} 
                          if isinstance(p, dict) and p.get("local_image_path") else None),
)
img = reg.render.map(
    spec,
    show_labels=True, show_country_names=True,
    show_pick_images=True, pick_image_size_px=125,
    show_flag_markers=False
)
display(img)


Keyword arguments {'add_watermark': False} are not expected by StableDiffusionXLPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

'artifacts/maps/Europe _ most popular musician _top-15 countries_.png'